# 05 — Comparison

Side-by-side comparison of this repo's unified pipeline against the four 2024 SIOP ML Competition winners. The intent is to make the tradeoffs concrete: where the unified approach gives up points to a hand-tuned bespoke approach, and where it's competitive.

## Headline scorecard

The winners' numbers are reconstructed from the official competition deck's per-task contribution table (each task contributes 0.25 × its raw metric to the composite). The **projected** column is the pre-run estimate against official inputs. The **measured (synthetic)** column is from a full test-split run on synthetic inputs (2026-05-31, `gpt-4o-2024-08-06`). **Do not compare measured synthetic scores to the winner columns** — see `docs/STATUS.md`.

| Task          | Metric           | PAID (1st) | Akben (2nd) | Hungry Llama (3rd) | Wonderlic (4th) | Projected | Measured (synthetic) |
|---------------|------------------|------------|-------------|---------------------|------------------|-----------|----------------------|
| Empathy       | accuracy         | .580       | **.608**    | .560                | .488             | ~.55–.60  | **1.000**            |
| Interview     | avg cosine       | .440       | .496        | **.512**            | .460             | ~.46–.50  | .309                 |
| Clarity       | Pearson r        | **.816**   | .676        | .740                | .772             | ~.65–.75  | .959                 |
| Fairness      | accuracy         | **.828**   | **.828**    | .760                | .792             | ~.78–.85  | **1.000**            |
| **Composite** | **0.25-weighted**| **.666**   | .652        | .643                | .630             | ~.61–.67  | **.817**             |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.scoring import composite

# Reconstruct each team's composite from their per-task scores.
# (This validates that our decoded numbers reproduce the published composites.)
teams = {
    "PAID":         {"e": 0.580, "i": 0.440, "c": 0.816, "f": 0.828},
    "Akben":        {"e": 0.608, "i": 0.496, "c": 0.676, "f": 0.828},
    "Hungry Llama": {"e": 0.560, "i": 0.512, "c": 0.740, "f": 0.760},
    "Wonderlic":    {"e": 0.488, "i": 0.460, "c": 0.772, "f": 0.792},
    "This repo (synthetic test)": {"e": 1.000, "i": 0.309, "c": 0.959, "f": 1.000},
}
print(f"{'Team':<28} {'Empathy':>8} {'Interview':>10} {'Clarity':>8} {'Fairness':>9} {'Composite':>10}")
print("-" * 79)
for name, s in teams.items():
    comp = composite(s["e"], s["i"], s["c"], s["f"])
    print(f"{name:<28} {s['e']:>8.3f} {s['i']:>10.3f} {s['c']:>8.3f} {s['f']:>9.3f} {comp:>10.3f}")

## Per-task narrative

### Empathy

| | What it does | Expected score |
|-|---|---|
| PAID 1st | GPT-4 + auto-generated reasoning chains as few-shot | .580 |
| Akben best | Ensemble: text-completion + Elo + label-learning, majority vote | **.608** |
| Hungry Llama | 7-dimensional Mistral/Mixtral ensemble | .560 |
| Wonderlic | SetFit + prompt tuning | .488 |
| This repo | GPT-4o + similarity-selected few-shot + structured outputs + optional SC | ~.55–.60 |

The unified approach should be competitive with PAID's .580 (similar architecture, modernized) and clearly behind Akben's .608 (we don't implement the Elo trick by default, though the code is in the notebook). Worth ~2-3 points if you add `--self-consistency 5`.

### Interview

| | What it does | Expected score |
|-|---|---|
| PAID 1st | GPT-4 question-centered (pooled responses) | .440 |
| Akben | GPT-4 + N candidates, pick highest cosine to input | .496 |
| Hungry Llama best | GPT-4 + Big-5 conditioning + reading level | **.512** |
| Wonderlic | Personality-aware prompting | .460 |
| This repo | GPT-4o + style-matching system prompt, 120-word cap | ~.46–.50 |

The unified approach should land near Akben/Wonderlic. To match Hungry Llama, you'd inject the personality/reading-level features (the cell is in notebook 02). To exceed them, layer Akben's cosine-reranking on top via `harness.call_consistent`.

### Clarity

| | What it does | Expected score |
|-|---|---|
| PAID best | **Fine-tuned DeBERTa-v3-base** | **.816** |
| Akben | Multi-model (GPT-4/3.5/Claude) ensemble across 15 binary sub-questions | .676 |
| Hungry Llama | Mixtral + BART Big-5 + NLP features → stacking | .740 |
| Wonderlic | (likely SetFit) | .772 |
| This repo (LLM-only) | GPT-4o + random few-shot | ~.65–.75 |

This is the only task where the unified approach has a real ceiling. To match PAID's .816 you must leave the harness and fine-tune DeBERTa (the training script is in notebook 03). The LLM-only ceiling is roughly r=.70-.75.

### Fairness

| | What it does | Expected score |
|-|---|---|
| PAID 1st (tied) | GPT-4 + auto-reason few-shot | **.828** |
| Akben 1st (tied) | GPT-4 + self-consistency on full few-shot | **.828** |
| Hungry Llama | Mixtral few-shot | .760 |
| Wonderlic | Prompt engineering / ICL | .792 |
| This repo | GPT-4o + full few-shot + structured outputs | ~.78–.85 |

The unified approach should match or slightly beat the .828 ceiling. The structured-output JSON mode eliminates parse errors that ate ~1-2 points for the 2024 winners; this is worth more than self-consistency on this task.

## What this experiment actually demonstrates

Three findings stand up across the four tasks:

1. **A single unified pipeline can match 3-of-4 winning approaches, but not all 4 simultaneously.** The harness should produce competitive results on empathy (.55-.60), interview (.46-.50), and fairness (.78-.85). It will leave 5-15 points on the table for clarity (.65-.75 vs. .816). The clarity gap is real and shows up because LLMs don't do calibrated regression well.

2. **The "right" technique varies by task in 2024, and it still does in 2026.** A 2026 reproduction with current models doesn't eliminate the need for task-specific reasoning. What it changes is:
   - Structured outputs (~1-2 points free)
   - Better instruction-following style transfer (~3-5 points free on interview)
   - Similarity-based few-shot (~2 points on empathy if you avoid Landmine 6)

3. **Two of the four winners' techniques generalize cleanly to other I-O tasks.** Akben's Elo-pairwise trick (for noisy human-rating tasks) and PAID's "use a regressor for regression" insight (for any continuous target) are both transferable. Hungry Llama's Big-5 conditioning and Wonderlic's SetFit are more situational.

## Composite score scenarios

Let's run the composite under different choices to see what each technique buys.

The "this repo, default" column uses GPT-4o, structured outputs, random few-shot, no self-consistency. The "this repo, optimized" column adds similarity-selected few-shot on empathy/fairness, self-consistency on empathy, Akben-style cosine reranking on interview, and switches clarity to fine-tuned DeBERTa.

Both are point estimates; treat ±.03 as the real uncertainty band.

In [ ]:
# Point-estimate scenarios (arithmetic + one measured row)
scenarios = {
    "PAID (actual winner)":       {"e": 0.580, "i": 0.440, "c": 0.816, "f": 0.828},
    "This repo, default (projected)": {"e": 0.575, "i": 0.475, "c": 0.700, "f": 0.815},
    "This repo, optimized (projected)": {"e": 0.605, "i": 0.510, "c": 0.815, "f": 0.835},
    "This repo, measured (synthetic test)": {"e": 1.000, "i": 0.309, "c": 0.959, "f": 1.000},
}

print(f"{'Scenario':<24} {'Emp':>5} {'Int':>5} {'Cla':>5} {'Fai':>5} {'Comp':>6}")
print("-" * 56)
for name, s in scenarios.items():
    comp = composite(s["e"], s["i"], s["c"], s["f"])
    print(f"{name:<24} {s['e']:>5.3f} {s['i']:>5.3f} {s['c']:>5.3f} {s['f']:>5.3f} {comp:>6.3f}")

print()
print("Notes:")
print("- 'Default' uses the unified harness with no optimizations beyond what's shipped on by default.")
print("- 'Optimized' adds: similarity few-shot (empathy/fairness), SC=5 (empathy), cosine rerank (interview),")
print("  and a separate DeBERTa fine-tune for clarity. Each of these is documented in its own notebook.")
print("- The 'optimized' composite is in PAID's territory because we're combining the best move from each")
print("  team into a single pipeline. This is the value proposition of the post-hoc reconstruction.")

## What's NOT in the unified pipeline

Three of the 2024 winners' techniques didn't make it into the harness, and not by accident:

1. **PAID's auto-reasoning step.** The cost (one extra API pass per training row) vs. the gain (~1 accuracy point) wasn't worth the implementation complexity for empathy/fairness specifically. With 2026 models the gain is even smaller.

2. **Hungry Llama's sub-dimension decomposition for empathy.** Seven separate LLM rating calls, weighted/summed into a final classification. The harness instead trusts GPT-4o to weigh the dimensions itself in a single call. The empirical evidence is that the modern model does as well or better.

3. **Wonderlic's SetFit/prompt-tuning.** Requires a separate training stack (sentence-transformers fine-tuning), and the empathy results suggest it doesn't reliably beat in-context learning when label noise is high. Not worth the integration complexity for a teaching repo.

If you wanted to fork this and add any of them, see `docs/ADAPTING_TO_NEW_TASKS.md` for the integration points.

## Closing thoughts

The 2024 competition shows the I-O ML field at a transition moment: LLMs are the default tool but not the right tool for every task. PAID won by recognizing this — they ran GPT-4 on three tasks and DeBERTa on the fourth. That heterogeneity is the actual lesson.

A "single unified pipeline" makes sense pedagogically (it's how I framed this repo) but is the wrong architecture for a real production system that cares about Pearson r on calibrated regression tasks. The harness here is closer to a teaching tool than a production system.

For someone planning a 2026 competition entry against a similar slate of tasks, the playbook from this comparison is:

1. **Default to GPT-4o + structured outputs + similarity few-shot.** This is the floor.
2. **Add Akben-style self-consistency** when the base model is uncertain (check by re-running at T=0.7 and counting flips). +1-3 points on classification.
3. **For any task with calibrated regression targets, fine-tune a BERT-family regressor.** Don't try to make the LLM do it.
4. **For generation tasks where similarity to a reference is the metric**, post-hoc rerank candidates by cosine to the input. Cheap and reliable.
5. **For tasks where the human label distribution is noisy**, pairwise comparison + Elo aggregation tends to beat direct rating. This is the move that's most underused outside the 2024 SIOP context.